In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp

base_path = "/Volumes/healthcare_adjudication/healthcare_adjudication_sch/healthcare_adjudication_vol"
bronze_schema = "healthcare_adjudication.healthcare_adjudication_bronze"

providers_schema = StructType([
    StructField("provider_id", IntegerType()),
    StructField("provider_name", StringType()),
    StructField("specialty", StringType()),
    StructField("state", StringType())
])

providers = (
    spark.read.format("csv")
    .option("header","true")
    .schema(providers_schema)
    .load(f"{base_path}/providers.csv")
    .withColumn("ingestion_time", current_timestamp())
)
providers.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{bronze_schema}.providers")


In [0]:
payers_schema = StructType([
    StructField("payer_id", IntegerType()),
    StructField("payer_name", StringType()),
    StructField("plan_type", StringType())
])

payers = (
    spark.read.format("csv")
    .option("header","true")
    .schema(payers_schema)
    .load(f"{base_path}/payers.csv")
    .withColumn("ingestion_time", current_timestamp())
)
payers.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{bronze_schema}.payers")


In [0]:
patients = (
    spark.read.option('multiLine', True).json(f"{base_path}/patients.json")
    .withColumn("ingestion_time", current_timestamp())
)
patients.write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.patients")


In [0]:
policies = (
    spark.read
    .option("multiline","true")
    .option("inferSchema","true")
    .json(f"{base_path}/policies_nested.json")
    .withColumn("ingestion_time", current_timestamp())
)
policies.write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.policies")


In [0]:
claim_lines = (
    spark.read
    .option("multiline","true")
    .option("inferSchema","true")
    .json(f"{base_path}/claim_lines.json")
    .withColumn("ingestion_time", current_timestamp())
)
claim_lines.write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.claim_lines")


In [0]:
claims_schema = StructType([
    StructField("claim_id", IntegerType()),
    StructField("patient_id", IntegerType()),
    StructField("provider_id", IntegerType()),
    StructField("policy_id", IntegerType()),
    StructField("claim_amount", DoubleType()),
    StructField("claim_date", DateType()),
    StructField("status", StringType())
])

claims = (
    spark.read.format("csv")
    .option("header","true")
    .schema(claims_schema)
    .load(f"{base_path}/claims.csv")
    .withColumn("ingestion_time", current_timestamp())
)
claims.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{bronze_schema}.claims")


In [0]:
payments = (
    spark.read
    .option("multiline","true")
    .option("inferSchema","true")
    .json(f"{base_path}/payments.json")
    .withColumn("ingestion_time", current_timestamp())
)
payments.write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.payments")
